In [1]:
import re
import pandas as pd
import yfinance as yf

In [18]:
# ==============================================================================
# 1. INPUT TANGGAL EVALUASI & RAW TEXT TRADING PLAN
# ==============================================================================
tanggal_evaluasi = "2026-09-7"  # Tanggal Acuan Evaluasi / Target Eksekusi (H+0)
JUMLAH_HARI_LOOKAHEAD = 10      # Evaluasi rentang H+0 s/d H+10 hari kalender

In [19]:
# TEMPEL SELURUH TEKS TRADING PLAN DI SINI
raw_text = """
==================================================
 TRADING PLAN: IMPC.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp1,500
. Cut Loss (-3%)    : Rp1,455
. Trailing Trigger  : Rp1,605 (+7%)
. Trailing Lock     : Rp1,575 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: INPC.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp178
. Cut Loss (-3%)    : Rp173
. Trailing Trigger  : Rp190 (+7%)
. Trailing Lock     : Rp187 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: MSIN.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp316
. Cut Loss (-3%)    : Rp307
. Trailing Trigger  : Rp338 (+7%)
. Trailing Lock     : Rp332 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: FILM.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp940
. Cut Loss (-3%)    : Rp912
. Trailing Trigger  : Rp1,006 (+7%)
. Trailing Lock     : Rp987 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: KKES.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp96
. Cut Loss (-3%)    : Rp93
. Trailing Trigger  : Rp103 (+7%)
. Trailing Lock     : Rp101 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: TOOL.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp71
. Cut Loss (-3%)    : Rp69
. Trailing Trigger  : Rp76 (+7%)
. Trailing Lock     : Rp75 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: CBRE.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp770
. Cut Loss (-3%)    : Rp747
. Trailing Trigger  : Rp824 (+7%)
. Trailing Lock     : Rp808 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: LAJU.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp75
. Cut Loss (-3%)    : Rp73
. Trailing Trigger  : Rp80 (+7%)
. Trailing Lock     : Rp79 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: FOLK.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp187
. Cut Loss (-3%)    : Rp181
. Trailing Trigger  : Rp200 (+7%)
. Trailing Lock     : Rp196 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: PPRI.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp163
. Cut Loss (-3%)    : Rp158
. Trailing Trigger  : Rp174 (+7%)
. Trailing Lock     : Rp171 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: TOSK.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp80
. Cut Loss (-3%)    : Rp78
. Trailing Trigger  : Rp86 (+7%)
. Trailing Lock     : Rp84 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: DATA.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-04 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-07 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp1,940
. Cut Loss (-3%)    : Rp1,882
. Trailing Trigger  : Rp2,076 (+7%)
. Trailing Lock     : Rp2,037 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================




"""


In [20]:
# ==============================================================================
# 2. PARSER TRADING PLAN (ANTI-GAGAL SPASI)
# ==============================================================================
def parse_plans(text):
    cleaned_text = text.replace('\xa0', ' ')
    blocks = cleaned_text.split("TRADING PLAN:")
    plans = []

    for block in blocks[1:]:
        ticker = re.search(r"^\s*([A-Za-z0-9\.]+)", block)
        buy = re.search(r"Harga Beli\s*Acuan\s*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        cut_loss = re.search(r"Cut Loss[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        lock = re.search(r"Trailing Lock[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)

        if ticker and buy and cut_loss and lock:
            code_raw = ticker.group(1).strip()
            ticker_formatted = code_raw if code_raw.endswith(".JK") else f"{code_raw}.JK"

            plans.append({
                "ticker": ticker_formatted,
                "code": ticker_formatted.replace(".JK", ""),
                "buy": int(buy.group(1).replace(",", "").replace(".", "")),
                "cut_loss": int(cut_loss.group(1).replace(",", "").replace(".", "")),
                "lock": int(lock.group(1).replace(",", "").replace(".", ""))
            })
    return plans

parsed_data = parse_plans(raw_text)

if not parsed_data:
    print("X Teks gagal dibaca. Pastikan format teks sesuai.")
else:
    df = pd.DataFrame(parsed_data)
    tickers = df["ticker"].tolist()

In [21]:
# ==============================================================================
# 3. AMBIL DATA DARI TANGGAL EVALUASI (H+0 s/d H+N)
# ==============================================================================
t_start = pd.to_datetime(tanggal_evaluasi)
t_end = t_start + pd.Timedelta(days=JUMLAH_HARI_LOOKAHEAD)

start_str = t_start.strftime('%Y-%m-%d')
end_str = t_end.strftime('%Y-%m-%d')

print(f"Membaca {len(tickers)} saham... Mengambil data dari {start_str} s/d {end_str}...")

# Download batch data historis
downloaded_data = yf.download(
    tickers,
    start=start_str,
    end=end_str,
    group_by='ticker',
    auto_adjust=False,
    progress=False
)

highest_high_list = []
lowest_low_list = []
tgl_open_list = []
status_list = []
tgl_tereksekusi_list = []

for idx, row in df.iterrows():
    ticker = row["ticker"]

    try:
        # Mengambil DataFrame spesifik per ticker
        df_tk = downloaded_data if len(tickers) == 1 else downloaded_data[ticker].dropna()

        # Jika ada MultiIndex level pada kolom
        if isinstance(df_tk.columns, pd.MultiIndex):
            df_tk.columns = df_tk.columns.get_level_values(0)

        if df_tk.empty:
            highest_high_list.append(None)
            lowest_low_list.append(None)
            tgl_open_list.append("-")
            status_list.append("DATA TIDAK ADA")
            tgl_tereksekusi_list.append("-")
            continue

        # Rekam High tertinggi & Low terendah selama periode
        highest_high_list.append(df_tk["High"].max())
        lowest_low_list.append(df_tk["Low"].min())

        # ==============================================================================
        # 4. CEK TANGGAL POSISI OPEN & TANGGAL EVALUASI TEREKSEKUSI
        # ==============================================================================
        tgl_open = "-"
        status = "HOLD"
        tgl_kejadian = "-"
        is_opened = False

        for date, price_row in df_tk.iterrows():
            date_str = date.strftime('%Y-%m-%d')

            # Cek kapan posisi Open pertama kali terjadi (Harga Low <= Harga Beli Acuan)
            if not is_opened and price_row["Low"] <= row["buy"]:
                tgl_open = date_str
                is_opened = True

            # Jika sudah pernah ter-open, cek status Profit / Loss
            if is_opened:
                # Prioritas 1: Menyentuh/Melewati Target Profit
                if price_row["High"] >= row["lock"]:
                    status = "PROFIT"
                    tgl_kejadian = date_str
                    break
                # Prioritas 2: Menyentuh/Turun ke Cut Loss
                elif price_row["Low"] <= row["cut_loss"]:
                    status = "LOSS"
                    tgl_kejadian = date_str
                    break

        tgl_open_list.append(tgl_open)
        status_list.append(status if is_opened else "BELUM OPEN")
        tgl_tereksekusi_list.append(tgl_kejadian)

    except Exception:
        highest_high_list.append(None)
        lowest_low_list.append(None)
        tgl_open_list.append("-")
        status_list.append("ERROR")
        tgl_tereksekusi_list.append("-")

# Masukkan hasil ke DataFrame
df["Highest High"] = highest_high_list
df["Lowest Low"] = lowest_low_list
df["Tanggal Posisi Open"] = tgl_open_list
df["Status Evaluasi"] = status_list
df["Tanggal Tereksekusi"] = tgl_tereksekusi_list

# ==============================================================================
# 5. TAMPILKAN HASIL EVALUASI
# ==============================================================================
print("\n=== HASIL EVALUASI TRADING PLAN ===")
cols_to_display = [
    "code", "buy", "cut_loss", "lock",
    "Highest High", "Lowest Low",
    "Tanggal Posisi Open", "Status Evaluasi", "Tanggal Tereksekusi"
]
display(df[cols_to_display])

Membaca 12 saham... Mengambil data dari 2026-09-07 s/d 2026-09-17...

=== HASIL EVALUASI TRADING PLAN ===


,code,buy,cut_loss,lock,Highest High,Lowest Low,Tanggal Posisi Open,Status Evaluasi,Tanggal Tereksekusi
0,IMPC,1500,1455,1575,1820.0,1450.0,2026-09-07,PROFIT,2026-09-07
1,INPC,178,173,187,192.0,151.0,2026-09-07,PROFIT,2026-09-07
2,MSIN,316,307,332,346.0,286.0,2026-09-07,PROFIT,2026-09-08
3,FILM,940,912,987,965.0,680.0,2026-09-07,LOSS,2026-09-07
4,KKES,96,93,101,101.0,82.0,2026-09-07,PROFIT,2026-09-07
5,TOOL,71,69,75,87.0,67.0,2026-09-07,PROFIT,2026-09-07
6,CBRE,770,747,808,780.0,695.0,2026-09-07,LOSS,2026-09-07
7,LAJU,75,73,79,80.0,67.0,2026-09-07,PROFIT,2026-09-07
8,FOLK,187,181,196,316.0,191.0,-,BELUM OPEN,-
9,PPRI,163,158,171,216.0,151.0,2026-09-07,PROFIT,2026-09-07
